# Setup

In [ ]:
import duckdb
import logging
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll
import splink.blocking_rule_library as brl

from splink.blocking_analysis import (
    count_comparisons_from_blocking_rules,
    chart_comparisons_from_blocking_rules,
    n_largest_blocks,
)

con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(connection=con)

df_l = db_api.register(df_l, table_name="df_l")
df_r = db_api.register(df_r, table_name="df_r")

In [ ]:
# have a look at the data
con.sql("select * from df_l")

# Blocking

In [ ]:
# add blocking rules here
brs = [
    
]

In [ ]:
chart_comparisons_from_blocking_rules(
    splink_dataframe_or_dataframes=[df_l, df_r],
    blocking_rules=brs,
    link_type='link_and_dedupe',
    record_sample_proportion=0.1,
)

In [ ]:
n_largest_blocks(
    splink_dataframe_or_dataframes=[df_l, df_r],
    blocking_rule=block_on("fname"),
    link_type='link_and_dedupe',
    n_largest=20,
).as_pandas_dataframe()

# Evaluation

In [ ]:
# Run these cells to calculate the recall of your blocking rules calculated using person_id.
# Obviously you can't do this for a real linkage!

# make each blocking rule stricter to only include true matches
brs_and_person_id = [
    brl.And(br, block_on("person_id"), arrays_to_explode=br.arrays_to_explode)
    for br in brs
]

settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[],
    blocking_rules_to_generate_predictions=brs_and_person_id,
    retain_intermediate_calculation_columns=False,
    retain_matching_columns=False,
)

linker = Linker([df_l, df_r], settings, log_level=20)

# deterministic link means we take every pair returned by the blocking rules as true links
df_pairs = linker.inference.deterministic_link()

res = con.sql(
    f"""
    select
        count(*) as count
    from {df_pairs.physical_name}
    """
)

found_matches_count = res.fetchall()[0][0]
print(f"Found matches = {found_matches_count}")

linker.table_management.delete_tables_created_by_splink_from_db()

In [ ]:
settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[],
    blocking_rules_to_generate_predictions=[block_on("person_id")],
    retain_intermediate_calculation_columns=False,
    retain_matching_columns=False,
)

linker = Linker([df_l, df_r], settings)

df_pairs = linker.inference.deterministic_link()

res = con.sql(
    f"""
    select
        count(*) as count
    from {df_pairs.physical_name}
    """
)

true_matches_count = res.fetchall()[0][0]

linker.table_management.delete_tables_created_by_splink_from_db()

print(f"Proportion of true matches found by blocking rules: {found_matches_count / true_matches_count}")